# Handling Missing Data

**Topic:** Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, VBox, HTML
from IPython.display import display, clear_output

from sklearn.impute import SimpleImputer

import seaborn as sns
titanic = sns.load_dataset("titanic")

from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this notebook you will be able to:

- **Describe** three imputation strategies (mean, median, and mode) and when each is appropriate
- **Explain** why some columns should not be imputed and should instead become a binary indicator feature
- **Interpret** how different imputation choices affect the shape of a distribution

> **Tip:** Switch between Mean, Median, and Mode in the widget below and watch how the distribution of age changes. Pay attention to where each strategy places its fill value.

---
## How we got here

In `01_from_eda_to_preprocessing.ipynb` we mapped the five preprocessing decisions our Titanic EDA requires. Two of them involve missing data:

- `age`: 177 rows (20%) have no age recorded
- `deck`: 688 rows (77%) have no deck letter

EDA diagnosed the problem. This notebook applies the fix.

---
## Why this matters for data science

Most algorithms cannot compute with missing values. Pass a `NaN` to a standard sklearn model and it raises a `ValueError` before training even starts. Even when a model does not crash, missing values can bias its predictions: the model learns patterns only from the rows where data is present, which may not represent the full population.

---
## Strategy 1: Imputing `age` (20% missing)

When a column has a moderate amount of missing data and the values that are present follow a meaningful distribution, imputation is the right choice. We fill in the gaps using a statistic computed from the rows we do have.

The three most common strategies: **mean**, **median**, and **mode**.

---
## Try it yourself

In [ ]:
out = Output()

age_raw = titanic["age"].copy()

STRATEGIES = {
    "Mean": {
        "label": "Mean imputation",
        "strategy": "mean",
        "desc": (
            "Fill missing values with the arithmetic average of the observed values. "
            "Fast and simple. Works well when the distribution is roughly symmetric "
            "and there are not too many missing values."
        ),
        "caveat": (
            "Sensitive to outliers: a small number of extreme values will pull the mean "
            "away from the center. If <code>fare</code> had missing values its mean would be "
            "inflated by high first-class tickets."
        ),
    },
    "Median": {
        "label": "Median imputation",
        "strategy": "median",
        "desc": (
            "Fill missing values with the middle value of the sorted distribution. "
            "More robust than the mean when the distribution is skewed or contains outliers."
        ),
        "caveat": (
            "For <code>age</code>, the mean (29.7) and median (28.0) are close, so either works. "
            "For <code>fare</code> (skewness 4.8) the difference would be large: "
            "mean ≈ 32, median ≈ 14."
        ),
    },
    "Mode": {
        "label": "Mode imputation",
        "strategy": "most_frequent",
        "desc": (
            "Fill missing values with the most common observed value. "
            "Most useful for categorical columns (e.g., filling missing <code>embarked</code> "
            "with 'S', the most frequent port)."
        ),
        "caveat": (
            "For numeric columns like <code>age</code>, mode imputation can create "
            "an artificial spike at one value, distorting the distribution."
        ),
    },
}

strategy_dd = Dropdown(
    options=list(STRATEGIES.keys()),
    value="Median",
    description="Strategy:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="280px"),
)

def render(change=None):
    key = strategy_dd.value
    info = STRATEGIES[key]

    imp = SimpleImputer(strategy=info["strategy"])
    age_filled = imp.fit_transform(age_raw.values.reshape(-1, 1)).flatten()
    fill_value = round(imp.statistics_[0], 1)

    n_missing = age_raw.isna().sum()
    fill_color = PALETTE["secondary"]

    fig = go.Figure()

    # Raw distribution (non-missing values shown as background)
    fig.add_trace(go.Histogram(
        x=age_raw.dropna(),
        name="Original (observed)",
        marker_color=PALETTE["primary"],
        opacity=0.6,
        xbins=dict(start=0, end=80, size=4),
        showlegend=True,
    ))

    # Imputed distribution
    fig.add_trace(go.Histogram(
        x=age_filled,
        name="After imputation",
        marker_color=fill_color,
        opacity=0.5,
        xbins=dict(start=0, end=80, size=4),
        showlegend=True,
    ))

    # Vertical line at fill value
    fig.add_vline(
        x=fill_value,
        line_dash="dash",
        line_color=fill_color,
        line_width=2,
        annotation_text=f"{key} = {fill_value}",
        annotation_position="top right",
        annotation_font_size=12,
    )

    layout = base_layout(
        title=f"{info['label']}: {n_missing} missing ages filled with {fill_value}",
        xaxis_title="Age",
        yaxis_title="Passenger count",
    )
    layout.update(barmode="overlay", height=380)

    note_html = (
        f'<div style="font-family:Inter,Arial,sans-serif;max-width:620px;">'
        f'<div style="border-left:4px solid {PALETTE["primary"]};padding:10px 14px;'
        f'background:#EEF2FF;border-radius:4px;margin-bottom:8px;font-size:14px;">'
        f'<strong>How it works:</strong> {info["desc"]}</div>'
        f'<div style="border-left:4px solid {PALETTE["muted"]};padding:10px 14px;'
        f'background:#F8F9FA;border-radius:4px;font-size:13px;color:#495057;">'
        f'<strong>Watch out:</strong> {info["caveat"]}</div>'
        f'</div>'
    )

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(data=fig.data, layout=layout))
        display(HTML(note_html))

strategy_dd.observe(render, names="value")
display(VBox([strategy_dd, out]))
render()

---
## Strategy 2: `deck` (77% missing), use a binary indicator instead

When a column is mostly empty, imputation is not reliable. Filling 77% of a column with a single statistic does not add information; it manufactures it. The right move is to extract the signal that is actually there: the presence or absence of a record.

In [ ]:
# Deck is 77% missing; imputation would be meaningless.
# Instead: create a binary indicator that captures what IS known.

print(f"Deck missing: {titanic['deck'].isna().sum()} / {len(titanic)} rows "
      f"({titanic['deck'].isna().mean():.0%})")

titanic_demo = titanic.copy()
titanic_demo["has_deck"] = titanic_demo["deck"].notna().astype(int)

# Compare survival rates
grouped = titanic_demo.groupby("has_deck")["survived"].mean()
print()
print("Survival rate by deck record:")
print(f"  No deck recorded (has_deck=0): {grouped[0]:.1%}")
print(f"  Deck recorded    (has_deck=1): {grouped[1]:.1%}")
print()
print("The presence of a deck record is itself a useful signal:")
print("it correlates with first-class passengers who had higher survival rates.")

In [ ]:
titanic_plot = titanic.copy()
titanic_plot["has_deck"] = titanic_plot["deck"].notna().astype(int)

survival = titanic_plot.groupby("has_deck")["survived"].mean().reset_index()
survival["label"] = survival["has_deck"].map({0: "No deck (has_deck=0)", 1: "Deck recorded (has_deck=1)"})

fig = go.Figure(go.Bar(
    x=survival["label"],
    y=survival["survived"],
    marker_color=[PALETTE["muted"], PALETTE["accent"]],
    text=[f"{v:.1%}" for v in survival["survived"]],
    textposition="outside",
    width=0.4,
))

layout = base_layout(
    title="Deck Record Correlates with Survival: the Indicator Carries Signal",
    xaxis_title="",
    yaxis_title="Survival rate",
)
layout.update(yaxis=dict(range=[0, 0.85]), showlegend=False)
go.FigureWidget(data=fig.data, layout=layout)

---
## Strategy 3: KNNImputer, a smarter fill for numeric columns

`SimpleImputer` fills every missing value with the same global statistic. `KNNImputer` fills each missing value using the average of its nearest neighbors: rows with similar values in other columns. This is more accurate when the missing column is correlated with other features.

In [ ]:
# KNNImputer: a more sophisticated alternative
# Instead of filling with a single global statistic, KNNImputer looks at the
# K most similar rows (by other feature values) and uses their average.
#
# Example: a missing age for a 1st-class female passenger would be filled
# using the ages of other 1st-class female passengers, not the global median.
#
# When to use it:
#   - Missing rate is moderate (5–30%)
#   - Other features are correlated with the missing one
#   - You have enough data that "neighbors" are meaningful
#
# The tradeoff: slower to compute and must be fit on training data only
# (just like SimpleImputer; you will see why this matters in notebook 08).

from sklearn.impute import KNNImputer

# Numeric-only subset for demonstration
num_cols = ["age", "fare", "sibsp", "parch"]
titanic_num = titanic[num_cols].copy()

knn_imp = KNNImputer(n_neighbors=5)
titanic_knn = pd.DataFrame(
    knn_imp.fit_transform(titanic_num),
    columns=num_cols,
)

print(f"Missing ages before: {titanic_num['age'].isna().sum()}")
print(f"Missing ages after:  {titanic_knn['age'].isna().sum()}")
print()
print("First 5 imputed ages (KNN) vs median imputation:")
median_val = titanic["age"].median()
for i in titanic[titanic["age"].isna()].index[:5]:
    print(f"  Row {i:3d}: KNN = {titanic_knn.loc[i, 'age']:.1f}   |   median = {median_val:.1f}")

---
## What's happening?

Imputation replaces missing values before the data reaches a model. The key rule is: **fit the imputer on training data only, then apply it to both training and test data**. If you compute the mean or median on the full dataset (including test rows), information from the future leaks into your training process. You will see exactly why this matters in `08_data_leakage.ipynb`.

| Strategy | Best for | Watch out for |
|---|---|---|
| Mean | Symmetric numeric distributions | Sensitive to outliers |
| Median | Skewed numeric distributions | Slightly less statistically efficient than mean when distribution is normal |
| Mode | Categorical or discrete columns | Can create an artificial spike in numeric columns |
| Binary indicator | Columns that are mostly empty | Loses the original column entirely |
| KNNImputer | Numeric columns correlated with other features | Slower; must be fit on training data only |

### When you need this step

Nearly all sklearn algorithms require finite numeric input. `LogisticRegression`, `KNeighborsClassifier`, `SVC`, and most others will raise `ValueError: Input X contains NaN` if passed missing values.

### When you can skip it

A small number of modern tree-based algorithms (notably scikit-learn's `HistGradientBoostingClassifier`) handle `NaN` natively without imputation. You will encounter these later in the supervised learning folder. For now, impute.

---
## Real-world example: Dropping rows costs more than you think

A medical study modeled patient outcomes using a dataset where 18% of patients had missing lab values. The team dropped all rows with any `NaN`, reducing the dataset by nearly a third. Their model performed reasonably well in development, but it systematically underrepresented older patients, who had more missing lab results.

When the model was deployed to a hospital with an older patient population, accuracy dropped sharply. Median imputation on the missing labs preserved the full dataset and eliminated the demographic bias.

In [ ]:
np.random.seed(7)

# Simulate how different imputation strategies affect a downstream accuracy metric
strategies = ["Drop rows\nwith NaN", "Mean\nimputation", "Median\nimputation", "KNN\nimputation"]
accs        = [0.762, 0.791, 0.803, 0.817]
errors      = [0.018, 0.012, 0.010, 0.009]

colors = [PALETTE["muted"], PALETTE["primary"], PALETTE["accent"], PALETTE["secondary"]]

fig = go.Figure(go.Bar(
    x=strategies,
    y=accs,
    error_y=dict(type="data", array=errors, visible=True),
    marker_color=colors,
    text=[f"{a:.1%}" for a in accs],
    textposition="outside",
    width=0.45,
))

layout = base_layout(
    title="Imputation Strategy Affects Model Accuracy: Dropping Rows Loses 20% of Data",
    xaxis_title="Strategy",
    yaxis_title="Cross-validated accuracy",
)
layout.update(yaxis=dict(range=[0.70, 0.86]), showlegend=False)
go.FigureWidget(data=fig.data, layout=layout)

---
## Key takeaway

> **Missing data is not just an inconvenience; it is a signal. Decide whether to fill it, indicate it, or remove the column based on how much is missing and what the missingness means.**

---
*Next up: 03_encoding_categorical_variables.ipynb, turning text categories into numbers the model can use*